In [1]:
import ee
import geemap
from dotenv import load_dotenv
import os
load_dotenv(dotenv_path='.env')
project_name = os.getenv("project_name")
# print(f"Project Name: {project_name}")
ee.Initialize(project=project_name)


In [2]:
map = geemap.Map()

point = ee.Geometry.Point([90.4152, 23.8041]) #around dhaka
region = ee.Geometry.Rectangle([90.3, 23.7, 90.5, 23.9]) #bounding box around the point

top_left = [23.822424724001266, 90.46289150228976]
bottom_right = [23.796369273111445, 90.5071668854189]

region = ee.Geometry.Rectangle([top_left[1], bottom_right[0], bottom_right[1], top_left[0]])

map.centerObject(point, 10)
map.addLayer(point, {'color': 'red'}, 'Point Layer')
map.addLayer(region, {'color': 'blue'}, 'Region Layer')

map #just testing if everything is working

Map(center=[23.8041, 90.4152], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topri…

In [3]:
def get_annual_image(year, aoi):
    start_date = ee.Date.fromYMD(year, 1, 1)
    end_date = ee.Date.fromYMD(year, 12, 31)
    
    collection = ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED") \
        .filterBounds(aoi) \
        .filterDate(start_date, end_date) \
        .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 5)) 
        
    return collection.median().clip(aoi).visualize(
        bands=['B4', 'B3', 'B2'],
        min=0,
        max=3000
    )


In [5]:
image_left = get_annual_image(2019, region)  # Before
image_right = get_annual_image(2024, region) # After

In [8]:
import geemap


#now we have to create the tile layers
#as we previously used vis, now we just give empty vis params {}
left_layer = geemap.ee_tile_layer(image_left, {}, '2019 View')
right_layer = geemap.ee_tile_layer(image_right, {}, '2024 View')

m = geemap.Map()
m.centerObject(region, 13)

m.split_map(
    left_layer=left_layer, 
    right_layer=right_layer
)

m

Map(center=[23.809398142431757, 90.48502919385766], controls=(ZoomControl(options=['position', 'zoom_in_text',…